In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import random
import numpy as np
from spatialca import net, map, game, viz
import os

## Data

In [ ]:
COLOUR_MAP = map.satellite_colour_map
num_classes = len(COLOUR_MAP)
_scale_down = 3
_n_splits = 3
num_agents = 20

_place = "vegas"
cities = [_place]
data_source = f"openearth_map_train_data_{_place}"
og_img_dir = f"./openearth/{_place}/images"
img_dir = f"./openearth/{_place}/labels"
data_file_path = f"./data/openearth_map_train_data_{_place}.json"

real_images = [f"{og_img_dir}/{f}" for f in os.listdir(og_img_dir)]
seg_images = [f"{img_dir}/{f}" for f in os.listdir(img_dir)]

real_image_paths = real_images[0:2]
train_image_paths = seg_images[0:2]
test_image_paths = seg_images[2:3]

In [ ]:
real_images = [map.load_rgb_image(image_path) for image_path in real_image_paths]
fig, axes = plt.subplots(1, len(real_images), figsize=(len(real_images) * 4, 4))

for i, rgb_image in enumerate(real_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Real {_place.capitalize()} {i + 1}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

seg_images = [
    map.load_openearth_seg_image(image_path, color_map=COLOUR_MAP)
    for image_path in train_image_paths
]
fig, axes = plt.subplots(1, len(seg_images), figsize=(len(seg_images) * 4, 4))

for i, rgb_image in enumerate(seg_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Seg. {_place.capitalize()} {i + 1}")

plt.tight_layout()
plt.show()

train_images = [
    map.load_openearth_seg_image(
        image_path, scale_down=_scale_down, color_map=COLOUR_MAP
    )
    for image_path in train_image_paths
]
train_images = [
    img for image in train_images for img in map.split_image(image, _n_splits)
]
rows = (len(train_images) + 7) // 9  # Calculate the number of rows needed
fig, axes = plt.subplots(rows, 9, figsize=(32, rows * 4))

for idx, rgb_image in enumerate(train_images):
    row, col = divmod(idx, 9)
    axes[row, col].imshow(rgb_image, cmap="gray")
    axes[row, col].set_title(f"Train Seg. {_place.capitalize()} {idx + 1}")

plt.tight_layout()
plt.show()

In [ ]:
COLOUR_MAP = map.synthetic_colours
num_classes = 4
num_agents = 10
cities = ["archipelago", "river", "farmland"]
data_source = "map_train_data_synth_all"

In [ ]:
place_to_eval = cities

n_cities = len(cities)
real_images = [
    map.generate_synthetic_satellite_image(city, COLOUR_MAP) for city in cities
]
fig, axes = plt.subplots(1, n_cities, figsize=(n_cities * 3, 3))
for i, rgb_image in enumerate(real_images):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(f"Map {i + 1}")
plt.tight_layout()
plt.show()

train_images = [
    map.generate_synthetic_satellite_image(city, COLOUR_MAP) for city in place_to_eval
]

## Models

In [ ]:
trained_models = {
    "UNet": net.UNet.load(f"best_chkpt_UNet-{data_source}.pth"),
    "NeuralCA.t=0": net.NeuralCA.load(f"best_chkpt_CNN-{data_source}.pth"),
    "NeuralCA.t=32": net.NeuralCA.load(f"best_chkpt_NeuralCA-{data_source}.pth"),
    # **{
    #     f"NeuralCA.t={t}": net.NeuralCA.load(
    #         f"best_chkpt_NeuralCA-t={t}-{data_source}.pth"
    #     )
    #     for t in [2, 5, 9, 13, 24, 32]
    # },
}

images_to_test = {
    "Train 1": train_images[0],
    "Train 2": train_images[1],
}
min_noise, max_noise = 0.1, 0.9

## Training Loss Curves

In [ ]:
loss_files = [x for x in os.listdir("./models") if x.startswith("losses")]
must_haves = []
if data_source:
    loss_files = [l_file for l_file in loss_files if data_source in l_file.lower()]
    if must_haves:
        loss_files = [
            l_file
            for l_file in loss_files
            if any(mh.lower() in l_file.lower() for mh in must_haves)
        ]

viz.plot_losses(loss_files, data_source, zoomed_xlim=1000, zoomed_ylims=(0.5, 1.5))

## Eval plots

In [ ]:
if len(cities) == 1:
    all_images = [
        map.load_openearth_seg_image(
            image_path, scale_down=_scale_down, color_map=COLOUR_MAP
        )
        for image_path in test_image_paths
    ]
    all_images = [
        img for image in all_images for img in map.split_image(image, _n_splits)
    ]
    _names = [f"{_place}-test-{i + 1}" for i in range(len(all_images))]
else:
    n_samples_per_city = 3
    _names = [f"{city}-{i + 1}" for city in cities for i in range(n_samples_per_city)]
    _config = map.SyntheticMapConfig()
    all_images = [
        map.generate_synthetic_satellite_image(c.split("-")[0], COLOUR_MAP, _config)
        for c in _names
    ]

In [ ]:
fig, axes = plt.subplots(1, len(all_images), figsize=(len(all_images) * 3, 3))
for i, (name, rgb_image) in enumerate(zip(_names, all_images)):
    axes[i].imshow(rgb_image, cmap="gray")
    axes[i].set_title(name)

plt.tight_layout()
plt.show()

## Analysis

In [ ]:
accs = {m: [] for m in trained_models.keys()}
ious = {m: [] for m in trained_models.keys()}

for j, sample_img in enumerate(all_images):
    map_snapshots = game.MultiAgentMapper(
        raw_map=torch.from_numpy(sample_img),
        num_agents=num_agents,
        movement_style="frontier",
        colour_map=COLOUR_MAP,
        agent_view_radius=2,
    ).collect_snapshot_data(
        rollouts=1,
        steps=150,
        interval=20,
    )
    map_snapshots = [
        (sample[0], sample[1])
        for sample in map_snapshots
        if min_noise < sample[0][:, :, 3].mean() < max_noise
    ]
    masked_samples = random.choices(map_snapshots, k=3)
    for i, (sample, target) in enumerate(masked_samples):
        seen, unseen = (
            (sample[:, :, -1] == 0).numpy(),
            (sample[:, :, -1] == 1).numpy(),
        )
        unseen_tens = torch.tensor(unseen)
        input_oh = map.rgb_to_input_tensor(sample, COLOUR_MAP)
        target_oh = map.rgb_to_target_tensor(target[:, :, :3], COLOUR_MAP)
        samples = input_oh.unsqueeze(0)

        for _model_name, model in trained_models.items():
            with torch.no_grad():
                # Predict the samples
                if isinstance(model, net.NeuralCA):
                    out_state = model(
                        logits=samples[:, :num_classes],
                        unseen_mask=samples[:, num_classes : num_classes + 1],
                        return_state=True,
                        return_probs=False,
                    )
                    preds = model.readout(out_state)  # (B, C, H, W)
                    probs = F.softmax(preds[0], dim=0)
                else:
                    preds = model(
                        logits=samples[:, :num_classes],
                        unseen_mask=samples[:, num_classes : num_classes + 1],
                        return_state=False,
                        return_probs=False,
                    )
                    probs = F.softmax(preds[0], dim=0)

            pred_labels = probs.argmax(dim=0).numpy()
            target_labels = target_oh.argmax(dim=0).numpy()

            # ---- Masked Accuracy ----
            correct = pred_labels == target_labels
            correct *= unseen
            acc = correct.sum() / unseen.sum()
            accs[_model_name].append(acc)

            # ---- Masked mIoU ----
            per_class_ious = []

            for c in range(num_classes):
                gt_c = (target_labels == c) & unseen
                pred_c = (pred_labels == c) & unseen

                if gt_c.sum() == 0:
                    continue  # skip classes not present in masked region

                tp = np.logical_and(pred_c, gt_c).sum()
                fp = np.logical_and(pred_c, ~gt_c).sum()
                fn = np.logical_and(~pred_c, gt_c).sum()

                denom = tp + fp + fn
                if denom > 0:
                    per_class_ious.append(tp / denom)

            # Mean over classes
            if len(per_class_ious) > 0:
                ious[_model_name].append(np.mean(per_class_ious))

            print("image", j, "repeat", i, _model_name)

In [ ]:
m_acc = {
    m: (
        100 * float(np.mean(values)),
        100 * float(np.std(values) * 1.96 / np.sqrt(len(values))),
    )
    for m, values in accs.items()
}
m_iou = {
    m: (float(np.mean(values)), float(np.std(values) * 1.96 / np.sqrt(len(values))))
    for m, values in ious.items()
}
for metric in ["Masked Accuracy", "Masked mIoU"]:
    print(metric)
    m_dict = m_acc if metric == "Masked Accuracy" else m_iou
    for m, (mean, ci) in m_dict.items():
        print(f"  {m}: {mean:.2f} ± {ci:.2f}")

In [ ]:
# test_img = map.generate_synthetic_satellite_image("archipelago", COLOUR_MAP)
test_img = all_images[1]

In [ ]:
map_snapshots = game.MultiAgentMapper(
    raw_map=torch.from_numpy(test_img),
    num_agents=num_agents,
    movement_style="frontier",
    colour_map=COLOUR_MAP,
    agent_view_radius=2,
).collect_snapshot_data(
    rollouts=1,
    steps=150,
    interval=20,
)
map_snapshots = [
    (sample[0], sample[1])
    for sample in map_snapshots
    if min_noise < sample[0][:, :, 3].mean() < max_noise
]

In [ ]:
sample, target = random.choice(map_snapshots)
seen, unseen = (
    (sample[:, :, -1] == 0).numpy(),
    (sample[:, :, -1] == 1).numpy(),
)
unseen_tens = torch.tensor(unseen)
input_oh = map.rgb_to_input_tensor(sample, COLOUR_MAP)
target_oh = map.rgb_to_target_tensor(target[:, :, :3], COLOUR_MAP)
samples = input_oh.unsqueeze(0)

In [ ]:
for _model_name, model in trained_models.items():
    print(f"{'-' * 20} {_model_name.upper()} {'-' * 20}")
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(sample[:, :, :3].numpy())
    axes[0].set_title("Sample")
    axes[0].axis("off")
    axes[1].imshow(target[:, :, :3].numpy())
    axes[1].set_title("Target")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    with torch.no_grad():
        # Predict the samples
        if isinstance(model, net.NeuralCA):
            out_state = model(
                logits=samples[:, :num_classes],
                unseen_mask=samples[:, num_classes : num_classes + 1],
                return_state=True,
                return_probs=False,
            )
            preds = model.readout(out_state)  # (B, C, H, W)
            probs = F.softmax(preds[0], dim=0)
        else:
            preds = model(
                logits=samples[:, :num_classes],
                unseen_mask=samples[:, num_classes : num_classes + 1],
                return_state=False,
                return_probs=False,
            )
            probs = F.softmax(preds[0], dim=0)

    # Convert preds to RGB
    rgb_predictions = map.softmax_to_rgb(probs, COLOUR_MAP)
    rgb_predictions = (
        target[:, :, :3].numpy() * seen[:, :, None]
        + rgb_predictions * unseen[:, :, None]
    )

    # Calculate entropy
    entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=0)
    entropy[seen] = entropy.min()
    e_min, e_max = entropy.min(), entropy.max()
    entropy = (entropy - e_min) / (e_max - e_min + 1e-8)

    # MSE
    error = ((probs - target_oh) ** 2).mean(dim=0)
    mean_err = error.sum(dim=(0, 1)) / unseen_tens.sum(dim=(0, 1))
    error *= unseen_tens

    correct = (target_oh.argmax(0) == probs.argmax(0)).float()
    correct[seen] = 0.5
    correct *= -1

    # Plot results
    fig, axes = plt.subplots(1, 4, figsize=(12, 4))

    axes[0].imshow(rgb_predictions)
    axes[0].set_title(f"{_model_name} - Prediction")
    axes[0].axis("off")

    axes[1].imshow(entropy, cmap="viridis")
    axes[1].set_title("Entropy")
    axes[1].axis("off")

    axes[2].imshow(error, cmap="inferno")
    axes[2].set_title(f"MSE {mean_err.item():.2f}")
    axes[2].axis("off")

    axes[3].imshow(correct, cmap="coolwarm")
    axes[3].set_title(f"Accuracy {correct.abs().mean().item():.2f}")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

## NeuralCA Videos

In [ ]:
sample, target = random.choice(map_snapshots)
seen, unseen = (
    (sample[:, :, -1] == 0).numpy(),
    (sample[:, :, -1] == 1).numpy(),
)
unseen_tens = torch.tensor(unseen)
input_oh = map.rgb_to_input_tensor(sample, COLOUR_MAP)
target_oh = map.rgb_to_target_tensor(target[:, :, :3], COLOUR_MAP)
samples = input_oh.unsqueeze(0)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample[:, :, :3].numpy())
axes[0].set_title("Sample")
axes[0].axis("off")
axes[1].imshow(target[:, :, :3].numpy())
axes[1].set_title("Target")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
model_to_test = trained_models["NeuralCA.t=32"]
interval = 500

In [ ]:
_state = samples.clone()


def update_frame(i, model_to_test):
    ax.clear()
    with torch.no_grad():
        global _state
        if i == 0:
            _state = model_to_test(
                logits=_state[:, :num_classes],
                unseen_mask=_state[:, num_classes : num_classes + 1],
                steps=1,
                return_state=True,
            )
        else:
            _state = model_to_test(current_state=_state, steps=1, return_state=True)
    _clamped_state = model_to_test.readout(_state.clone())
    rgb_out = map.softmax_to_rgb(_clamped_state[0], COLOUR_MAP)
    rgb_out = target[:, :, :3].numpy() * seen[:, :, None] + rgb_out * unseen[:, :, None]
    ax.imshow(rgb_out)
    ax.set_title(f"Step {i + 1}")
    ax.axis("off")
    return (ax,)


# Set up the figure first
fig, ax = plt.subplots(figsize=(7, 7))

# Create the animation
ani = animation.FuncAnimation(
    fig,
    update_frame,
    frames=model_to_test.full_steps,
    fargs=(model_to_test,),
    interval=interval,
)

ani.save("./videos/neural_ca_example.gif", writer="pillow")

In [ ]:
_state = samples.clone()


def update_frame(i, model_to_test):
    ax.clear()
    with torch.no_grad():
        global _state
        if i == 0:
            _state = model_to_test(
                logits=_state[:, :num_classes],
                unseen_mask=_state[:, num_classes : num_classes + 1],
                steps=1,
                return_state=True,
            )
        else:
            _state = model_to_test(current_state=_state, steps=1, return_state=True)
    probs = model_to_test.softmax(model_to_test.readout(_state.clone()))[0]
    entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=0)  # Shape: [H, W]
    entropy = entropy * unseen_tens
    avg_entropy = entropy.sum() / unseen_tens.sum()
    entropy_max, entropy_min = entropy.max(), entropy[unseen_tens].min()
    ax.imshow(entropy.detach().numpy(), cmap="inferno", vmin=0)
    ax.set_title(
        f"Step {i + 1}, Entropy: {avg_entropy:.4f} [{entropy_min:.4f}, {entropy_max:.4f}]"
    )
    ax.axis("off")
    return (ax,)


# Set up the figure first
fig, ax = plt.subplots(figsize=(7, 7))

# Create the animation
ani = animation.FuncAnimation(
    fig,
    update_frame,
    frames=model_to_test.full_steps,
    fargs=(model_to_test,),
    interval=interval,
)

ani.save("./videos/neural_ca_entropy.gif", writer="pillow")

In [ ]:
_state = samples.clone()
states = []


def update_frame(i, model_to_test):
    ax.clear()
    global _state, states
    with torch.no_grad():
        if i == 0:
            _state = model_to_test(
                logits=_state[:, :num_classes],
                unseen_mask=_state[:, num_classes : num_classes + 1],
                steps=1,
                return_state=True,
            )
        else:
            _state = model_to_test(current_state=_state, steps=1, return_state=True)
    # Save current state for diff tracking
    states.append(_state.clone())
    # Compute diff from previous state (if exists)
    diff = states[-1][0].mean(dim=0)
    average_diff = diff.mean().item()
    diff_max, diff_min = diff.max(), diff[unseen_tens].min()
    ax.imshow(diff.numpy(), cmap="viridis")
    ax.set_title(
        f"Step {i + 1}, dx: {average_diff:.4f} [{diff_min:.4f}, {diff_max:.4f}]"
    )
    ax.axis("off")
    return (ax,)


fig, ax = plt.subplots(figsize=(7, 7))
ani = animation.FuncAnimation(
    fig,
    update_frame,
    frames=model_to_test.full_steps,
    fargs=(model_to_test,),
    interval=interval,
)
ani.save("./videos/neural_ca_state.gif", writer="pillow")

In [ ]:
_state = samples.clone()
states = []


def update_frame(i, model_to_test):
    ax.clear()
    global _state, states
    with torch.no_grad():
        if i == 0:
            _state = model_to_test(
                logits=_state[:, :num_classes],
                unseen_mask=_state[:, num_classes : num_classes + 1],
                steps=1,
                return_state=True,
            )
        else:
            _state = model_to_test(current_state=_state, steps=1, return_state=True)
    # Save current state for diff tracking
    states.append(_state.clone())
    # Compute diff from previous state (if exists)
    if i > 0:
        diff = (states[-1] - states[-2])[0].mean(dim=0)
    else:
        diff = torch.zeros_like(_state[0, 0])
    average_diff = diff.mean().item()
    diff_max, diff_min = diff.max(), diff[unseen_tens].min()
    ax.imshow(diff.numpy(), cmap="viridis")
    ax.set_title(
        f"Step {i + 1}, dx: {average_diff:.4f} [{diff_min:.4f}, {diff_max:.4f}]"
    )
    ax.axis("off")
    return (ax,)


fig, ax = plt.subplots(figsize=(7, 7))
ani = animation.FuncAnimation(
    fig,
    update_frame,
    frames=model_to_test.full_steps,
    fargs=(model_to_test,),
    interval=interval,
)
ani.save("./videos/neural_ca_dx.gif", writer="pillow")

In [ ]:
states_to_plot = states

In [ ]:
# ---------- 1. Compute mean activation over time ----------
stack = torch.stack([s[0] for s in states_to_plot], dim=0)  # [T, C, H, W]
mean_state_np = stack.mean(dim=0).mean(dim=0).numpy()  # [H, W]

# ---------- 2. Compute temporal average gradient field ----------
grads_x = []
grads_y = []
for s in states_to_plot:
    frame = s[0].mean(dim=0)  # [H, W]
    gy, gx = torch.gradient(frame)
    grads_x.append(-gx)
    grads_y.append(-gy)

avg_gx = torch.stack(grads_x, dim=0).mean(dim=0)
avg_gy = torch.stack(grads_y, dim=0).mean(dim=0)

# Streamlines
H, W = mean_state_np.shape
Y, X = np.mgrid[0:H, 0:W]

# Other
mask = samples.clone()[0, -1].bool()
abs_act = stack.abs().mean(dim=1).flatten(1)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
# ax.imshow(mean_state_np, cmap="viridis", alpha=0.8, origin="upper")
ax.imshow(sample[:, :, :3].numpy())
ax.streamplot(
    X,
    Y,
    avg_gx.numpy(),
    avg_gy.numpy(),
    density=1.0,
    color="white",
    linewidth=1.0,
    arrowsize=1.0,
)
ax.axis("off")
plt.tight_layout()
# plt.savefig(
#     "./videos/neuralca_message_passing_streamlines.png", dpi=300, bbox_inches="tight"
# )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
H, W = states[0][0].shape[1:]  # assume [C,H,W] format
Y, X = np.mgrid[0:H, 0:W]


def update_frame(t):
    ax.clear()
    frame_mean = states[t][0].mean(0).numpy()
    gx = grads_x[t].numpy()
    gy = grads_y[t].numpy()
    ax.imshow(frame_mean, cmap="viridis", alpha=0.8, origin="upper")
    ax.streamplot(
        X, Y, gx, gy, color="white", density=1.0, linewidth=1.0, arrowsize=1.0
    )
    ax.axis("off")
    ax.set_title(f"Timestep {t + 1}")
    return (ax,)


# Create animation
ani = animation.FuncAnimation(
    fig,
    update_frame,
    frames=len(states),
    interval=interval,
)
ani.save("./videos/neuralca_message_passing.gif", writer="pillow")

In [ ]:
# --- Sample ---
fig = plt.figure(figsize=(4, 4))
plt.imshow(sample[:, :, :3].numpy())
plt.axis("off")
plt.tight_layout()
plt.show()
# fig.savefig("./paper/assets/dynamics/sample.png", dpi=300, bbox_inches="tight")

# --- Target ---
fig = plt.figure(figsize=(4, 4))
plt.imshow(target[:, :, :3].numpy())
plt.axis("off")
plt.tight_layout()
plt.show()
# fig.savefig("./figures/target.png", dpi=300, bbox_inches='tight')

# --- Prediction ---
fig = plt.figure(figsize=(4, 4))
plt.imshow(rgb_predictions)
plt.axis("off")
plt.tight_layout()
plt.show()
# fig.savefig(
#     f"./paper/assets/dynamics/{_model_name.lower()}_prediction.png",
#     dpi=300,
#     bbox_inches="tight",
# )